# WTI Crude Oil Price Forecasting — Progressive Agentic Introduction

This notebook introduces the progressive capability staircase for agentic forecasting models. We will study a single, high-stakes forecasting origin date (March 2, 2026), right as the Persian Gulf escalation begins and shipping lane closures in the Strait of Hormuz are announced.

This workbook demonstrates the 4-step progressive escalation:
1. **Step 1: The Blind Statistical Baseline**: Fits a standard Prophet model that is blind to real-world geopolitics.
2. **Step 2: Basic Agentic Predictor**: Employs an LLM direct-prompting predictor (no search or tools) that must guess based on price tables alone.
3. **Step 3: News-Grounded Agent**: Equips the agent with a `context_agent` tool (bounded news search with strict temporal cutoffs) to retrieve real-world intelligence briefings.
4. **Step 4: Advanced Code-Executing Agent**: Leverages **Gemini's Native Code Execution** tool alongside custom forecasting skills to write Python code, analyze volatility, fit regressions, and visually check charts inline before predicting.

---
## 1. Setup & Data Registration

We register the continuous front-month futures contract `CL=F` from Yahoo Finance under the series ID `wti_crude_oil_price` using the `DataService` and the `YFinanceDailyAdapter` included in the `aieng` library.

In [ ]:
import os
import json
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings("ignore")

from aieng.forecasting.data import DataService, SeriesMetadata
from aieng.forecasting.data.adapters.yfinance import YFinanceDailyAdapter

# Setup DataService and register WTI daily close series
data_service = DataService()
wti_adapter = YFinanceDailyAdapter(ticker="CL=F", field="Close")
data_service.register(
    "wti_crude_oil_price",
    wti_adapter,
    SeriesMetadata(
        series_id="wti_crude_oil_price",
        description="WTI Crude Oil Close price (Yahoo Finance CL=F)",
        source="yfinance",
        units="USD/bbl",
        frequency="B",
    )
)

# Fetch history up to March 2, 2026
origin = pd.Timestamp("2026-03-02")
as_of = origin - pd.Timedelta(days=1)
ctx = data_service.context(as_of=as_of)
full_df = ctx.get_series("wti_crude_oil_price")
print(f"Total historical trading days available up to {as_of.date()}: {len(full_df)}")
print(f"Closing price on March 2, 2026: ${full_df['value'].iloc[-1]:.2f}/bbl")

---
## 2. Step 1: The Blind Statistical Baseline (Prophet)

We fit a Prophet baseline on WTI price history. Prophet extrapolates from trends and seasonal patterns, but lacks the structural knowledge to anticipate regime changes caused by real-world geopolitical conflicts.

In [ ]:
from prophet import Prophet

# Prepare history for Prophet (needs columns 'ds' and 'y')
train_df = full_df.rename(columns={"timestamp": "ds", "value": "y"})
train_df["ds"] = pd.to_datetime(train_df["ds"])

model = Prophet(
    seasonality_mode="multiplicative",
    changepoint_prior_scale=0.1,
    changepoint_range=0.9
)
model.fit(train_df)

# Forecast 30 calendar days ahead (Prophet is daily, so we generate calendar days and map to business days)
future = model.make_future_dataframe(periods=30, freq="D")
forecast = model.predict(future)

# Inspect predictions at horizons 5, 10, and 21 business days
prophet_pred = forecast.set_index("ds")
print("Prophet Forecast Trajectory (as of 2026-03-02):")
for h in [5, 10, 21]:
    pred_date = origin + pd.Timedelta(days=h)
    # Snap to nearest in index
    idx_date = prophet_pred.index[prophet_pred.index >= pred_date][0]
    row = prophet_pred.loc[idx_date]
    print(f"  Day {h:>2} ({idx_date.date()}): Point=${row['yhat']:.2f}  Interval=[${row['yhat_lower']:.2f}, ${row['yhat_upper']:.2f}]")

---
## 3. Step 2: Basic Agentic Predictor (Direct Prompting)

We construct a direct-prompting Analyst Agent (using the Boring baseline). This model only sees the numerical close price history as a raw text string, with no additional tools or news search.

In [ ]:
import litellm

_ANALYST_SYSTEM = (
    "You are an expert oil market analyst.\n\n"
    "You will receive:\n"
    "  1. WTI crude oil price history (daily, most recent last)\n"
    "  2. An oil market intelligence briefing with a strict temporal cutoff\n"
    "  3. A TASK SPECIFICATION that defines the exact question and the required\n"
    "     JSON output schema\n\n"
    "Read the data and briefing carefully, then execute the task precisely.\n"
    "Return ONLY valid JSON that matches the schema described in the task specification.\n"
    "Output ONLY the JSON object — no preamble, no explanation outside the JSON."
)

TASK_TRAJECTORY = (
    "Forecast the WTI crude oil price at three forward horizons from today:\n"
    "  - 5  business days (~1 trading week)\n"
    "  - 10 business days (~2 trading weeks)\n"
    "  - 21 business days (~1 calendar month)\n\n"
    "For each horizon provide a point estimate and an 80% confidence interval.\n"
    "Be calibrated: historical weekly vol is roughly $2-5/bbl; widen intervals\n"
    "when the market is unusually uncertain.\n\n"
    "Return JSON with exactly these fields:\n"
    "{\n"
    '  "day_5":           <float>,\n'
    '  "lower_80_day_5":  <float>,\n'
    '  "upper_80_day_5":  <float>,\n'
    '  "day_10":          <float>,\n'
    '  "lower_80_day_10": <float>,\n'
    '  "upper_80_day_10": <float>,\n'
    '  "day_21":          <float>,\n'
    '  "lower_80_day_21": <float>,\n'
    '  "upper_80_day_21": <float>,\n'
    '  "reasoning":       "<2-4 sentences>",\n'
    '  "confidence":      "<high|medium|low>"\n'
    "}"
)

def compress_history(df: pd.DataFrame, max_rows=40) -> str:
    subset = df.iloc[-max_rows:]
    return "\n".join(f"{pd.Timestamp(r['timestamp']).strftime('%Y-%m-%d')} ${r['value']:.2f}" for _, r in subset.iterrows())

hist_str = compress_history(full_df)
origin_price = full_df["value"].iloc[-1]

user_prompt_basic = (
    f"### WTI Price History (ending 2026-03-02)\n\n{hist_str}\n\n"
    f"---\n### Oil Market Briefing (as of 2026-03-02)\n\nNone available (no tools enabled).\n\n"
    f"---\n### TASK SPECIFICATION\n\n{TASK_TRAJECTORY}\n\n"
    f"Current WTI price: ${origin_price:.2f}/bbl on 2026-03-02."
)

response_basic = litellm.completion(
    model="gemini/gemini-3-flash-preview",
    messages=[
        {"role": "system", "content": _ANALYST_SYSTEM},
        {"role": "user", "content": user_prompt_basic},
    ],
    temperature=0.2,
    response_format={"type": "json_object"},
)
print("Basic Direct-Prompted Agent (No News) Forecast:")
print(response_basic.choices[0].message.content)

---
## 4. Step 3: News-Grounded Agentic Predictor (Context Retrieval)

Now we equip the agent with a `context_agent` tool (bounded news search via Google ADK with `google_search` tool). We enforce a strict temporal cutoff of **March 1, 2026** (one day before the origin date) to retrieve real-world intelligence briefings in a backtest-safe manner.

In [ ]:
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search
from google.genai import types as genai_types
from google.genai.types import GenerateContentConfig

_CONTEXT_SYSTEM = (
    "You are an oil market intelligence specialist with access to web search.\n\n"
    "CRITICAL TEMPORAL CONSTRAINT — you are simulating the perspective of an analyst\n"
    "as of {cutoff_date}.\n"
    "- Include ONLY information publicly available BEFORE {cutoff_date}.\n"
    "- EXCLUDE any events, market moves, or data from {cutoff_date} or later.\n"
    "- If a search result post-dates the cutoff, skip it entirely.\n\n"
    "Search for and summarise:\n"
    "- WTI/Brent crude price level and recent trend\n"
    "- OPEC+ production decisions and supply outlook\n"
    "- Geopolitical risks in the Persian Gulf, Middle East, key shipping lanes\n"
    "- US Strategic Petroleum Reserve and energy policy signals\n"
    "- Notable tanker/shipping incidents or supply chain disruption signals\n"
    "- Published analyst forecasts or unusual price-target revisions\n\n"
    "Return a concise structured markdown summary (3-5 paragraphs)."
)

async def retrieve_context(cutoff_date: str) -> str:
    _app = "oil-context-agent"
    session_svc = InMemorySessionService()
    agent = LlmAgent(
        name="oil_context_agent",
        instruction=_CONTEXT_SYSTEM.format(cutoff_date=cutoff_date),
        tools=[google_search],
        model="gemini-3-flash-preview",
        generate_content_config=GenerateContentConfig(temperature=0.1, max_output_tokens=4096),
    )
    runner = Runner(agent=agent, app_name=_app, session_service=session_svc)
    session = await session_svc.create_session(app_name=_app, user_id="nb")
    prompt = (
        f"Oil market intelligence briefing as of {cutoff_date}. "
        f"Focus on supply risks, OPEC+ policy, Persian Gulf geopolitics, and factors "
        f"that could cause a sudden large price move in WTI. "
        f"IMPORTANT: only use information available before {cutoff_date}."
    )
    content = genai_types.Content(role="user", parts=[genai_types.Part(text=prompt)])
    async for event in runner.run_async(user_id="nb", session_id=session.id, new_message=content):
        if event.is_final_response() and event.content and event.content.parts:
            return event.content.parts[0].text or ""
    return ""

# Run the Context Agent sync using asyncio
import asyncio
briefing = asyncio.run(retrieve_context("2026-03-01"))

print("━" * 72)
print("RETRIEVED CONTEMPORANEOUS BRIEFING as of March 1, 2026:")
print("━" * 72)
print(briefing)

Now we pass this retrieved news context into our Analyst Agent. The agent can now reason over the escalations in the Strait of Hormuz and OPEC+ cuts, adjusting its trajectory projections upward accordingly.

In [ ]:
user_prompt_grounded = (
    f"### WTI Price History (ending 2026-03-02)\n\n{hist_str}\n\n"
    f"---\n### Oil Market Briefing (as of 2026-03-02)\n\n{briefing}\n\n"
    f"---\n### TASK SPECIFICATION\n\n{TASK_TRAJECTORY}\n\n"
    f"Current WTI price: ${origin_price:.2f}/bbl on 2026-03-02."
)

response_grounded = litellm.completion(
    model="gemini/gemini-3-flash-preview",
    messages=[
        {"role": "system", "content": _ANALYST_SYSTEM},
        {"role": "user", "content": user_prompt_grounded},
    ],
    temperature=0.2,
    response_format={"type": "json_object"},
)
print("News-Grounded Agent Forecast:")
print(response_grounded.choices[0].message.content)

---
## 5. Step 4: Code-Executing Agentic Predictor (Gemini Native)

In this configuration, we enable **Gemini's Native Code Execution** tool inside the model's GenerateContentConfig, allowing it to write and execute Python code iteratively inside an isolated sandbox.

To support high-quality forecasting, we define **3 basic, introductory skills** within its prompts to guide its computational analysis:
1.  **Data Aggregation & Noise Reduction (using `pandas` and `numpy`)**: Write code to load daily WTI history, align trading days, compute rolling metrics (e.g., 5-day and 20-day Simple Moving Averages) to smooth noise, and compute rolling standard deviation (volatility) to understand recent market turbulence.
2.  **Trend Projection & Calibration (using `scikit-learn` or `scipy`)**: Write code to fit a simple linear regression or polynomial trend on the most recent 30 trading days, project point forecasts to the horizons, and calibrate prediction intervals using residual standard errors.
3.  **Inline Plotting and Visual Sanity Checks (using `matplotlib`)**: Write code to generate a chart displaying prices, rolling averages, projected lines, and shaded 80% confidence bands. Because Gemini returns these matplotlib plots as inline response images, the agent can "visually inspect" its own forecasts, checking for trend implausibility or interval misalignment before returning its structured JSON.

In [ ]:
from google import genai
from google.genai import types

# Construct combined prompt with the 3 Forecasting Skills
SKILLS_INSTRUCTION = (
    "### Custom Forecasting Skills (Executable Python Environment)\n"
    "You are equipped with a sandboxed Python runtime containing pandas, numpy, scipy, scikit-learn, and matplotlib. \n"
    "You can generate and run code iteratively using the code_execution tool to analyze numerical trends.\n"
    "Use these 3 specific forecasting skills before making your final prediction:\n\n"
    "1. DATA AGGREGATION & NOISE REDUCTION:\n"
    "   Write code to convert the daily close series to a pandas DataFrame, and compute a 5-day and 20-day Simple Moving Average (SMA) or Exponentially Weighted Moving Average (EWMA) to smooth noise and verify if the trend is accelerating. Compute rolling 10-day historical standard deviation to understand the base rate of market volatility.\n\n"
    "2. TREND PROJECTION & QUANTILE CALIBRATION:\n"
    "   Write code to fit a quick linear regression or polynomial trend (using scikit-learn or numpy) on the most recent 30 trading days. Project this trend out to 5, 10, and 21 trading days ahead. Compute the standard error of the residuals on the fitted history to calibrate your 80% confidence interval width:\n"
    "   - Margin of Error (80% CI) = 1.28 * residual_std_error * sqrt(horizon_days / 5)\n\n"
    "3. INLINE PLOTTING & VISUAL SANITY CHECKS:\n"
    "   Write code using matplotlib to plot the daily closing price, moving averages, projected linear trend, and shaded 80% confidence intervals. The chart will render inline. Visually inspect the chart to ensure your forecasted intervals are calibrated and that your point forecasts do not physically violate market bounds before printing your final JSON."
)

user_prompt_code = (
    f"### WTI Price History (ending 2026-03-02)\n\n{hist_str}\n\n"
    f"---\n### Oil Market Briefing (as of 2026-03-02)\n\n{briefing}\n\n"
    f"---\n### FORECASTING SKILLS\n\n{SKILLS_INSTRUCTION}\n\n"
    f"---\n### TASK SPECIFICATION\n\n{TASK_TRAJECTORY}\n\n"
    f"Current WTI price: ${origin_price:.2f}/bbl on 2026-03-02.\n"
    f"IMPORTANT: You MUST write Python code to analyze the data, plot your forecast, and use the results to produce your final structured JSON prediction."
)

# Initialize native GenAI client (reads GEMINI_API_KEY from env)
client = genai.Client()

response_code = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=user_prompt_code,
    config=types.GenerateContentConfig(
        system_instruction=_ANALYST_SYSTEM,
        tools=[types.Tool(code_execution=types.ToolCodeExecution())],  # Enable native code execution
        temperature=0.2
    ),
)

# Print each part of the response (text, code, execution results, and inline plots!)
print("━" * 72)
print("INTERACTIVE CODE EXECUTION TIMELINE:")
print("━" * 72)
for part in response_code.candidates[0].content.parts:
    if part.text is not None:
        print("\n--- [Text Response] ---")
        print(part.text)
    if part.executable_code is not None:
        print("\n--- [Generated Python Code] ---")
        print(part.executable_code.code)
    if part.code_execution_result is not None:
        print("\n--- [Execution Output] ---")
        print(part.code_execution_result.output)